# Micro CNN Model for Binary Classification

This notebook contains the micro CNN model with ~150 parameters. This model is specifically designed for small datasets and eliminates the FC bottleneck using global average pooling.


In [5]:
import pandas as pd 
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


In [6]:
class MicroCNN(nn.Module):
    """Small CNN optimized for small datasets - slightly larger than micro for better learning"""
    def __init__(self, input_channels: int = 4, sequence_length: int = 600):
        super(MicroCNN, self).__init__()
        # Small but more capable architecture
        self.conv1 = nn.Conv1d(input_channels, 8, kernel_size=7, stride=1, padding=3)
        self.conv2 = nn.Conv1d(8, 16, kernel_size=5, stride=1, padding=2)
        self.pool = nn.MaxPool1d(4)  # 600 -> 150 -> 37.5 ≈ 37
        
        # Calculate size after convolutions and pooling
        # After 2 pooling operations: 600 // (4*4) = 600 // 16 = 37.5 ≈ 37
        self.final_length = sequence_length // 16
        
        # Small FC layers with global pooling option
        self.fc1 = nn.Linear(16 * self.final_length, 32)  # More capacity
        self.fc2 = nn.Linear(32, 1)
        self.dropout = nn.Dropout(0.4)  # Slightly higher dropout

    def forward(self, x):
        # x shape: [batch, length, channels] -> transpose to [batch, channels, length]
        x = x.transpose(1, 2)  # [batch, channels, length]
        x = self.pool(F.relu(self.conv1(x)))  # [batch, 8, 150]
        x = self.pool(F.relu(self.conv2(x)))  # [batch, 16, 37]
        
        # Flatten for fully connected layers
        x = x.view(x.size(0), -1)  # [batch, 16*37 = 592]
        x = self.dropout(F.relu(self.fc1(x)))  # [batch, 32]
        x = torch.sigmoid(self.fc2(x))  # [batch, 1]
        
        return x


In [7]:
# Load and prepare data
data_binary = pd.read_csv('../../data/processed/ProSeq_binary_classification.csv')
data_binary = data_binary[['binary_classification', 'ProSeq']]

print(f"Original dataset size: {len(data_binary)}")
sequence_lengths = data_binary['ProSeq'].str.len()
print(f"Sequence length range: {sequence_lengths.min()} to {sequence_lengths.max()}")

# Keep only sequences with length >= 600
data_filtered = data_binary[sequence_lengths >= 600].copy()
print(f"Filtered dataset size (length >= 600): {len(data_filtered)}")

class BinaryClassificationDataset(Dataset):
    """Dataset for binary classification."""
    def __init__(self, data, target_length=600):
        self.data = data
        self.dna_dict = {"A": 0, "T": 1, "G": 2, "C": 3}
        self.target_length = target_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sequence = self.data.iloc[idx]['ProSeq']
        target = self.data.iloc[idx]['binary_classification']
        one_hot_sequence = self.one_hot_encode(sequence)
        return one_hot_sequence, target
    
    def one_hot_encode(self, sequence):
        # Truncate sequence to target_length (all sequences are >= target_length)
        sequence = sequence[:self.target_length]
        
        one_hot = np.zeros((self.target_length, 4), dtype=np.float32)
        for i, nucleotide in enumerate(sequence):
            if nucleotide in self.dna_dict:
                one_hot[i, self.dna_dict[nucleotide]] = 1.0
        return one_hot

# Split data
train_data, test_data = train_test_split(data_filtered, test_size=0.2, random_state=42)
train_data, val_data = train_test_split(train_data, test_size=0.2, random_state=42)

# Create datasets and dataloaders
train_dataset = BinaryClassificationDataset(train_data)
val_dataset = BinaryClassificationDataset(val_data)
test_dataset = BinaryClassificationDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")


Original dataset size: 8307
Sequence length range: 472.0 to 600.0
Filtered dataset size (length >= 600): 8302
Train samples: 5312
Val samples: 1329
Test samples: 1661


In [8]:
# Train Micro CNN model
micro_model = MicroCNN(input_channels=4)
criterion = nn.BCELoss()
optimizer = optim.Adam(micro_model.parameters(), lr=0.001, weight_decay=1e-3)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
micro_model.to(device)

print(f"Training on device: {device}")
print(f"Micro model parameters: {sum(p.numel() for p in micro_model.parameters()):,}")
print(f"Samples per parameter ratio: {len(data_filtered) / sum(p.numel() for p in micro_model.parameters()):.1f}")

# Break down parameters by layer
print("\nParameter breakdown:")
conv_params = sum(p.numel() for name, p in micro_model.named_parameters() if 'conv' in name)
fc_params = sum(p.numel() for name, p in micro_model.named_parameters() if 'fc' in name)
print(f"  Conv layer: {conv_params:,} parameters")
print(f"  FC layer: {fc_params:,} parameters")
print(f"  Total: {conv_params + fc_params:,} parameters")

# Train with early stopping
NUM_EPOCHS = 100
train_losses = []
val_losses = []
best_val_loss = float('inf')
patience = 15
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    # Training phase
    micro_model.train()
    epoch_train_loss = 0.0
    for batch_idx, batch in enumerate(train_loader):
        x, y = batch
        if epoch == 0 and batch_idx == 0:
            print(f"\nInput shape: {x.shape}, Target shape: {y.shape}")
            
        x = x.to(device)
        y = y.float().to(device)
        
        optimizer.zero_grad()
        output = micro_model(x)
        output = output.squeeze()
        
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        
        epoch_train_loss += loss.item()
    
    # Validation phase
    micro_model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            x, y = batch
            x = x.to(device)
            y = y.float().to(device)
            
            output = micro_model(x)
            output = output.squeeze()
            
            val_loss = criterion(output, y)
            epoch_val_loss += val_loss.item()
    
    # Calculate average losses
    avg_train_loss = epoch_train_loss / len(train_loader)
    avg_val_loss = epoch_val_loss / len(val_loader)
    
    # Store losses
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    
    # Early stopping check
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        best_model_state = micro_model.state_dict().copy()
    else:
        patience_counter += 1
    
    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{NUM_EPOCHS} - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')
    
    # Early stopping
    if patience_counter >= patience:
        print(f'Early stopping at epoch {epoch+1}')
        micro_model.load_state_dict(best_model_state)
        break

print(f"\nTraining completed. Best validation loss: {best_val_loss:.4f}")


Training on device: cpu
Micro model parameters: 19,897
Samples per parameter ratio: 0.4

Parameter breakdown:
  Conv layer: 888 parameters
  FC layer: 19,009 parameters
  Total: 19,897 parameters
✅ Excellent ratio for small dataset!

Input shape: torch.Size([32, 600, 4]), Target shape: torch.Size([32])
Epoch 10/100 - Train Loss: 0.6609, Val Loss: 0.6568
Epoch 20/100 - Train Loss: 0.6604, Val Loss: 0.6562
Epoch 30/100 - Train Loss: 0.6590, Val Loss: 0.6573
Early stopping at epoch 35

Training completed. Best validation loss: 0.6562
